In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline


class NeighborhoodClusterer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=3, demographic_cols=None):
        self.n_clusters = n_clusters
        self.demographic_cols = demographic_cols
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)

    def fit(self, X, y=None):
        demo_data = X[self.demographic_cols]
        scaled_demo = self.scaler.fit_transform(demo_data)
        self.kmeans.fit(scaled_demo)
        return self

    def transform(self, X):
        X_out = X.copy()
        demo_data = X_out[self.demographic_cols]
        scaled_demo = self.scaler.transform(demo_data)
        X_out['Neighborhood_Cluster'] = self.kmeans.predict(scaled_demo)
        X_out = X_out.drop(columns=self.demographic_cols)
        return X_out

np.random.seed(42)
n_samples = 1000

property_data = pd.DataFrame({
    'sq_foot': np.random.normal(2100, 450, n_samples),
    'property_age': np.random.randint(1, 60, n_samples),
    'bedrooms': np.random.randint(2, 6, n_samples),
    'bathrooms': np.random.randint(1, 4, n_samples),
    'crime_rate': np.random.uniform(0.1, 4.5, n_samples),
    'avg_income': np.random.normal(70000, 18000, n_samples),
    'school_rating': np.random.uniform(2, 10, n_samples),
    'transit_score': np.random.uniform(15, 85, n_samples)
})

base_price = (property_data['sq_foot'] * 165) - (property_data['property_age'] * 780) + (property_data['avg_income'] * 1.4)
property_data['price'] = base_price + np.random.normal(0, 12000, n_samples)

credit_data = pd.DataFrame({
    'credit_score': np.random.randint(580, 850, n_samples),
    'debt_to_income': np.random.uniform(0.15, 0.55, n_samples),
    'down_payment_pct': np.random.uniform(0.05, 0.30, n_samples)
})
risk_formula = (credit_data['debt_to_income'] * 7.5) - (credit_data['credit_score'] / 155) - (credit_data['down_payment_pct'] * 3.5)
credit_data['default'] = (1 / (1 + np.exp(-risk_formula)) > np.random.uniform(0, 1, n_samples)).astype(int)
# Outlier removal for property prices
Q1, Q3 = property_data['price'].quantile(0.25), property_data['price'].quantile(0.75)
IQR = Q3 - Q1
clean_property_df = property_data[(property_data['price'] >= (Q1 - 1.5 * IQR)) & (property_data['price'] <= (Q3 + 1.5 * IQR))].copy()

X_prop = clean_property_df.drop(columns=['price'])
y_prop = clean_property_df['price']

demographic_features = ['crime_rate', 'avg_income', 'school_rating', 'transit_score']
physical_features = ['sq_foot', 'property_age', 'bedrooms', 'bathrooms']

valuation_pipeline = Pipeline([
    ('unsupervised_segmentation', NeighborhoodClusterer(n_clusters=3, demographic_cols=demographic_features)),
    ('encoding_and_scaling', ColumnTransformer(transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Neighborhood_Cluster']),
        ('num', StandardScaler(), physical_features)
    ])),
    ('linear_regressor', LinearRegression())
])
valuation_pipeline.fit(X_prop, y_prop)

property_data['predicted_house_price'] = valuation_pipeline.predict(property_data.drop(columns=['price', 'predicted_house_price'], errors='ignore'))
X_credit = credit_data[['credit_score', 'down_payment_pct']].copy()
X_credit['predicted_house_price'] = property_data['predicted_house_price']
y_credit = credit_data['default']

classification_pipeline = Pipeline([
    ('feature_scaler', StandardScaler()),
    ('random_forest', RandomForestClassifier(max_depth=6, n_estimators=100, random_state=42))
])
classification_pipeline.fit(X_credit, y_credit)


def assess_investment_deal(property_features, applicant_credit_profiles):
    input_property_df = pd.DataFrame([property_features])
    suggested_valuation = valuation_pipeline.predict(input_property_df)[0]

    credit_input_matrix = pd.DataFrame([
        {
            'credit_score': applicant_credit_profiles['credit_score'],
            'down_payment_pct': applicant_credit_profiles['down_payment_pct'],
            'predicted_house_price': suggested_valuation
        }
    ])

    raw_default_prob = classification_pipeline.predict_proba(credit_input_matrix)[0][1]
    decision_threshold = 0.25
    underwriting_status = "DENIED (Elevated Default Risk Exposure)" if raw_default_prob >= decision_threshold else "APPROVED"

    return {
        "Suggested Price": f"${suggested_valuation:,.2f}",
        "Risk Probability": f"{raw_default_prob * 100:.2f}%",
        "Approval Status": underwriting_status
    }


def get_dynamic_user_input():
    print("\n" + "="*50)
    print("      REAL ESTATE & CREDIT RISK RISK ENGINE")
    print("="*50)
    print("Please enter the requested values below:")

    try:

        print("\n--- 1. Property Structural Metrics ---")
        sq_ft = float(input("Enter Square Foot (e.g., 2000): "))
        age = int(input("Enter Property Age in Years (e.g., 5): "))
        beds = int(input("Enter Number of Bedrooms (e.g., 3): "))
        baths = int(input("Enter Number of Bathrooms (e.g., 2): "))

        print("\n--- 2. Neighborhood Socioeconomic Data ---")
        crime = float(input("Enter Local Crime Rate Index (0.1 - 5.0): "))
        income = float(input("Enter Average Neighborhood Local Income ($): "))
        school = float(input("Enter Public School Rating (1.0 - 10.0): "))
        transit = float(input("Enter Proximity to Transit Score (0 - 100): "))


        print("\n--- 3. Applicant Financial Underwriting Metrics ---")
        credit = int(input("Enter Applicant Credit Score (300 - 850): "))
        down_pmt = float(input("Enter Down Payment Percentage as decimal (e.g., 0.10 for 10%): "))


        property_payload = {
            'sq_foot': sq_ft, 'property_age': age, 'bedrooms': beds, 'bathrooms': baths,
            'crime_rate': crime, 'avg_income': income, 'school_rating': school, 'transit_score': transit
        }

        applicant_payload = {
            'credit_score': credit,
            'down_payment_pct': down_pmt
        }


        deal_result = assess_investment_deal(property_payload, applicant_payload)

        print("\n" + "-"*40)
        print("          UNDERWRITING ASSESSMENT")
        print("-"*40)
        for key, value in deal_result.items():
            print(f"{key:<20}: {value}")
        print("-"*40)

    except ValueError:
        print("\n[ERROR]: Invalid entry detected. Please ensure numerical features are inputted correctly.")

if __name__ == "__main__":

    while True:
        get_dynamic_user_input()
        cont = input("\nEvaluate another deal? (y/n): ").strip().lower()
        if cont != 'y':
            print("\nExiting Engine Execution Loop.")
            break



      REAL ESTATE & CREDIT RISK RISK ENGINE
Please enter the requested values below:

--- 1. Property Structural Metrics ---
